# 07 — Procesamiento de datos: Parseo de archivos IDF

La capa `Tools` proporciona utilidades para parsear los archivos **IDF (Ivium Data Format)** propietarios de Ivium. Funciona completamente **sin conexión** — no se necesita IviumSoft, hardware ni driver.

### Qué puedes hacer

- Parsear archivos IDF en listas/dicts de Python
- Exportar datos a CSV
- Convertir en lote un directorio completo de archivos IDF
- Visualizar datos con pandas y matplotlib

### Estructura del archivo IDF

Un archivo IDF puede contener múltiples secciones de datos:

| Clave de sección | Contenido |
|-------------|--------|
| `primary_data` | Medición principal (siempre presente) — p.ej. pares E/I para LSV/CV, tiempo/I/E para transitorios, Z1/Z2/frec para EIS |
| `ocpdata` | Medición OCP registrada antes del experimento |
| `pretreatmentdata` | Datos del paso de pretratamiento |
| `RsCs_data` | Datos de medición Rs/Cs |
| `osc_data` | Datos de osciloscopio (lista de secciones, una por sub-barrido) |

### Datos de prueba

Este notebook usa archivos IDF de `tests/data/` — están confirmados en el repositorio para que puedas ejecutar este notebook ahora mismo.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from pyvium import Tools

# Localizar datos de prueba relativos a este notebook
REPO_ROOT = Path("..")
TEST_DATA  = REPO_ROOT / "tests" / "data"

EIS_FILE  = TEST_DATA / "eis_test.idf"
OPEN_FILE = TEST_DATA / "test_open.idf"

print("Available test files:")
for f in TEST_DATA.iterdir():
    print(f"  {f.name}  ({f.stat().st_size / 1024:.1f} KB)")

## 1. `get_idf_data()` — Solo datos primarios

La función más sencilla: extrae solo la sección `primary_data` y la devuelve como una lista plana de filas.

In [ ]:
primary = Tools.get_idf_data(str(EIS_FILE))

print(f"Type    : {type(primary)}")
print(f"Points  : {len(primary)}")
print(f"Row[0]  : {primary[0]}")
print(f"Row[-1] : {primary[-1]}")

# Cada fila es una lista de flotantes — el significado de la columna depende del tipo de método
# Para EIS: [Re(Z), Im(Z), frecuencia]
# Para LSV: [potencial, corriente, 0]
# Para transitorios: [tiempo, corriente, potencial]

## 2. `get_all_idf_data()` — Todas las secciones

Devuelve un diccionario con todas las secciones presentes en el archivo. Las claves que no existen en el archivo simplemente están ausentes del dict.

In [ ]:
all_data = Tools.get_all_idf_data(str(EIS_FILE))

print(f"Sections found: {list(all_data.keys())}")
for section, rows in all_data.items():
    if rows:
        print(f"  {section}: {len(rows)} point(s)")

## 3. Datos EIS — Diagrama de Nyquist

Las filas de `primary_data` EIS son `[Re(Z), Im(Z), frecuencia]`.

In [ ]:
eis_df = pd.DataFrame(
    Tools.get_idf_data(str(EIS_FILE)),
    columns=["Z_re", "Z_im", "Frequency"]
)
print(eis_df.head())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Diagrama de Nyquist
axes[0].plot(eis_df["Z_re"], -eis_df["Z_im"], 'o-', markersize=4)
axes[0].set_xlabel("Z' (Ω)")
axes[0].set_ylabel("-Z'' (Ω)")
axes[0].set_title("Nyquist Plot")
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

# Diagrama de Bode — |Z| vs frecuencia
import numpy as np
Z_mag = np.sqrt(eis_df["Z_re"]**2 + eis_df["Z_im"]**2)
phase = np.degrees(np.arctan2(-eis_df["Z_im"], eis_df["Z_re"]))

ax_mag = axes[1]
ax_mag.semilogx(eis_df["Frequency"], Z_mag, 'b-o', markersize=3, label='|Z|')
ax_mag.set_xlabel("Frequency (Hz)")
ax_mag.set_ylabel("|Z| (Ω)", color='b')
ax_mag.set_title("Bode Plot")
ax_mag.grid(True, alpha=0.3)

ax_phase = ax_mag.twinx()
ax_phase.semilogx(eis_df["Frequency"], phase, 'r--o', markersize=3, label='Phase')
ax_phase.set_ylabel("Phase (°)", color='r')

plt.tight_layout()
plt.show()

## 4. Datos OCP

Si el archivo IDF incluye una pre-medición OCP, aparece en la clave `ocpdata`. Las filas son típicamente `[tiempo, potencial, corriente]`.

In [ ]:
all_data = Tools.get_all_idf_data(str(EIS_FILE))

if "ocpdata" in all_data and all_data["ocpdata"]:
    ocp_df = pd.DataFrame(all_data["ocpdata"])
    print(f"OCP data: {len(ocp_df)} points")
    print(ocp_df.head())

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(ocp_df.iloc[:, 0], ocp_df.iloc[:, 1], 'g-')
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("OCV (V)")
    ax.set_title("Open Circuit Potential (pre-measurement)")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No OCP data in this file")

## 5. Inspeccionar todas las secciones

In [ ]:
for section, rows in all_data.items():
    if not rows:
        print(f"  {section}: empty")
        continue

    # osc_data es una lista de sub-secciones
    if section == "osc_data":
        print(f"  {section}: {len(rows)} sub-section(s)")
        for i, sub in enumerate(rows):
            print(f"    sub-section {i}: {len(sub)} points, {len(sub[0]) if sub else 0} columns")
    else:
        cols = len(rows[0]) if rows else 0
        print(f"  {section}: {len(rows)} points, {cols} columns")

## 6. Exportar a CSV

`export_to_csv(data, path)` escribe cualquier lista de filas en un archivo CSV.

In [ ]:
import tempfile

# Exportar datos primarios a CSV
csv_path = os.path.join(tempfile.gettempdir(), "eis_primary.csv")
Tools.export_to_csv(primary, csv_path)
print(f"Exported to: {csv_path}")

# Verificar leyendo de vuelta
readback = pd.read_csv(csv_path, header=None)
print(f"Read back: {len(readback)} rows × {len(readback.columns)} columns")
print(readback.head())

## 7. `convert_idf_to_csv()` — Conversión en una línea

Lee un archivo IDF, extrae los datos primarios y escribe un archivo `.idf.csv` junto al archivo fuente — todo en una llamada.

In [ ]:
import shutil

# Trabajar en una copia para no modificar los datos de prueba
tmp_dir  = Path(tempfile.mkdtemp())
tmp_idf  = tmp_dir / "eis_test.idf"
shutil.copy(EIS_FILE, tmp_idf)

Tools.convert_idf_to_csv(str(tmp_idf))

csv_out = tmp_idf.with_suffix(".idf.csv")
print(f"Created: {csv_out.name}  ({csv_out.stat().st_size} bytes)")

# Vista previa
df_check = pd.read_csv(csv_out, header=None)
print(df_check.head())

## 8. `convert_idf_dir_to_csv()` — Conversión en lote

Convierte cada archivo `.idf` en un directorio. Devuelve un dict de resumen con conteos y los nombres de archivos que fallaron.

In [ ]:
# Copiar todos los archivos IDF de prueba a un directorio temporal
batch_dir = Path(tempfile.mkdtemp())
for idf in TEST_DATA.glob("*.idf"):
    shutil.copy(idf, batch_dir / idf.name)

print(f"Converting {len(list(batch_dir.glob('*.idf')))} IDF files in {batch_dir}")

result = Tools.convert_idf_dir_to_csv(str(batch_dir))

print(f"\nResults:")
print(f"  Converted : {result['converted_count']}")
print(f"  Errors    : {result['error_count']}")
if result['errors']:
    print(f"  Failed    : {result['errors']}")

print("\nOutput files:")
for f in sorted(batch_dir.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size} bytes)")

## 9. Construir una tubería de análisis

Ejemplo: procesar un directorio de archivos EIS y extraer una sola cantidad (|Z| a 1 Hz) de cada uno.

In [ ]:
import numpy as np
from pathlib import Path

def extract_z_at_freq(idf_path: str, target_freq: float) -> float:
    """Devuelve |Z| a la frecuencia más cercana a target_freq."""
    data = Tools.get_idf_data(idf_path)
    df = pd.DataFrame(data, columns=["Z_re", "Z_im", "Frequency"])
    idx = (df["Frequency"] - target_freq).abs().idxmin()
    row = df.iloc[idx]
    return np.sqrt(row["Z_re"]**2 + row["Z_im"]**2)

# Ejecutar en todos los archivos IDF del directorio de prueba
results = []
for idf_file in sorted(TEST_DATA.glob("*.idf")):
    try:
        z = extract_z_at_freq(str(idf_file), target_freq=1.0)
        results.append({"file": idf_file.name, "|Z| at 1 Hz (Ω)": z})
    except Exception as e:
        results.append({"file": idf_file.name, "|Z| at 1 Hz (Ω)": None})
        print(f"  Skipped {idf_file.name}: {e}")

summary = pd.DataFrame(results)
print(summary.to_string(index=False))

---

## Resumen

| Tarea | Método | Notas |
|------|--------|------|
| Leer datos primarios | `Tools.get_idf_data(path)` | Devuelve `List[List[float]]` |
| Leer todas las secciones | `Tools.get_all_idf_data(path)` | Devuelve `Dict[str, List]` |
| Escribir CSV | `Tools.export_to_csv(data, path)` | Cualquier lista de filas |
| Convertir un archivo | `Tools.convert_idf_to_csv(path)` | Crea `.idf.csv` junto al archivo fuente |
| Convertir en lote un directorio | `Tools.convert_idf_dir_to_csv(dir)` | Devuelve dict de resumen |

### Significado de las columnas de la sección de datos

| Sección | Col 0 | Col 1 | Col 2 |
|---------|-------|-------|-------|
| `primary_data` (EIS) | Re(Z) Ω | Im(Z) Ω | Frecuencia Hz |
| `primary_data` (LSV/CV) | Potencial V | Corriente A | 0 |
| `primary_data` (transitorio) | Tiempo s | Corriente A | Potencial V |
| `ocpdata` | Tiempo s | Potencial V | Corriente A |

## Siguiente

- **`08_batch_and_synchronization.ipynb`** — Coordinar múltiples dispositivos con el parámetro de estado